# Fine-tune an existing colony model on newly reviewed plates

Starts from an existing model and teaches it newly labelled plates, mixed with some older plates so it doesn't forget them. It then tests the old and the new model side by side on held-out plates that neither trained on.

**Upload** `finetune_data.zip`, laid out as:

```
data/train/images, data/train/labels       new plates + some older "reminder" plates
data/val_new/images, data/val_new/labels   held-out new plates
data/val_old/images, data/val_old/labels   held-out older plates
data/base_v2.pt                            the model to start from (a copy of app/model/best.pt)
```

**Before you start:** `Runtime -> Change runtime type -> T4 GPU -> Save`, then `Runtime -> Run all` and upload the zip at step 2. A whole run takes about 20-30 minutes; training itself is about 5 minutes for 130 plates.

At the end the browser downloads **finetune_result.zip** (`best_finetuned.pt`, `compare.json`, `results.csv`). Check the side-by-side table before replacing `app/model/best.pt`, and keep the old model as a backup.

## 1. GPU and YOLO

In [ ]:
!nvidia-smi -L
%pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 2. Upload finetune_data.zip

In [ ]:
import os, zipfile, shutil, glob, json
from google.colab import files

shutil.rmtree('/content/ft', ignore_errors=True)
print('Choose finetune_data.zip ...')
up = files.upload()
name = next(k for k in up if k.lower().endswith('.zip'))
zipfile.ZipFile(name).extractall('/content/ft')
os.remove(name)

D = '/content/ft/data'
for part in ['train', 'val_new', 'val_old']:
    imgs = glob.glob(f'{D}/{part}/images/*')
    labs = glob.glob(f'{D}/{part}/labels/*.txt')
    print(f'{part:8s} images {len(imgs):4d}  labels {len(labs):4d}')
    assert imgs and len(imgs) == len(labs), f'{part}: images and labels do not match'
assert os.path.exists(f'{D}/base_v2.pt'), 'base_v2.pt missing from the zip'

## 3. Dataset files

In [ ]:
def write_yaml(path, train, val):
    vals = '\n'.join(f'  - {v}' for v in val)
    with open(path, 'w') as f:
        f.write(f'path: {D}\ntrain: {train}\nval:\n{vals}\nnames:\n  0: colony\n')

# Training picks its best epoch on new AND old held-out plates,
# so a model that forgets the old plates doesn't win.
write_yaml('/content/ft/train.yaml', 'train/images', ['val_new/images', 'val_old/images'])
write_yaml('/content/ft/val_new.yaml', 'val_new/images', ['val_new/images'])
write_yaml('/content/ft/val_old.yaml', 'val_old/images', ['val_old/images'])
print(open('/content/ft/train.yaml').read())

## 4. Test the current model (v2.0)

Two kinds of score on the held-out plates:
- **mAP50 / recall / precision**: YOLO's standard detection scores (higher is better).
- **Count error**: run it like the app does (confidence 0.25) and compare the
  number of colonies found with your labels.

In [ ]:
from ultralytics import YOLO

SETTINGS = dict(imgsz=1024, max_det=3000)

def count_test(model, folder):
    rows = []
    for p in sorted(glob.glob(f'{folder}/images/*')):
        stem = os.path.splitext(os.path.basename(p))[0]
        truth = sum(1 for l in open(f'{folder}/labels/{stem}.txt') if l.strip())
        r = model.predict(p, conf=0.25, iou=0.45, verbose=False, **SETTINGS)[0]
        rows.append([stem, truth, len(r.boxes)])
    return rows

def evaluate(weights, tag):
    m = YOLO(weights)
    out = {}
    for part in ['val_new', 'val_old']:
        v = m.val(data=f'/content/ft/{part}.yaml', batch=8, plots=False, verbose=False,
                  project='/content/runs', name=f'val_{tag}_{part}', exist_ok=True, **SETTINGS)
        rows = count_test(m, f'{D}/{part}')
        big = [r for r in rows if r[1] >= 10]
        out[part] = dict(
            mAP50=float(v.box.map50), mAP50_95=float(v.box.map),
            recall=float(v.box.mr), precision=float(v.box.mp),
            colonies_labelled=sum(r[1] for r in rows),
            colonies_found=sum(r[2] for r in rows),
            mean_abs_error=sum(abs(r[2] - r[1]) for r in rows) / len(rows),
            within_10pct=sum(abs(r[2] - r[1]) <= 0.1 * r[1] for r in big) / max(len(big), 1),
            plates=rows)
    return out

before = evaluate(f'{D}/base_v2.pt', 'v2')
for part, s in before.items():
    print(part, {k: round(v, 3) if isinstance(v, float) else v for k, v in s.items() if k != 'plates'})

## 5. Fine-tune

Starts from v2.0. The first 10 layers (the part that recognises basic shapes)
stay frozen and the learning rate is low, so it adjusts what it knows rather
than starting over. Stops early if the held-out score stops improving.

In [ ]:
model = YOLO(f'{D}/base_v2.pt')
model.train(
    data='/content/ft/train.yaml',
    epochs=40,
    patience=15,
    imgsz=1024,
    batch=8,
    optimizer='AdamW',
    lr0=0.0005,
    warmup_epochs=1,
    freeze=10,
    copy_paste=0.3,
    max_det=3000,
    project='/content/runs',
    name='finetune',
    exist_ok=True,
)
run_dir = str(model.trainer.save_dir)
best_pt = os.path.join(run_dir, 'weights', 'best.pt')
if not os.path.exists(best_pt):
    best_pt = sorted(glob.glob('/content/runs/**/finetune*/weights/best.pt', recursive=True))[-1]
print('Best weights:', best_pt)

## 6. Old vs new, side by side

In [ ]:
after = evaluate(best_pt, 'new')

rows = [('mAP50', 'mAP50'), ('recall', 'recall'), ('precision', 'precision'),
        ('colonies found', 'colonies_found'), ('mean error (colonies/plate)', 'mean_abs_error'),
        ('plates within 10%', 'within_10pct')]
for part, title in [('val_new', 'NEW plates (22 held out)'), ('val_old', 'OLD plates (20 held out)')]:
    print(f'\n{title}   labelled colonies: {before[part]["colonies_labelled"]}')
    print(f'  {"":30s}{"v2.0":>10s}{"fine-tuned":>12s}')
    for label, k in rows:
        a, b = before[part][k], after[part][k]
        fmt = (lambda x: f'{x:.0%}') if k == 'within_10pct' else (lambda x: f'{x:.3f}' if isinstance(x, float) else str(x))
        print(f'  {label:30s}{fmt(a):>10s}{fmt(b):>12s}')

print('\nPer-plate counts on the new held-out plates (labelled / v2.0 / fine-tuned):')
for (s, t, a), (_, _, b) in zip(before['val_new']['plates'], after['val_new']['plates']):
    print(f'  {s[:12]:12s} {t:4d} {a:5d} {b:5d}')

## 7. Download the result

In [ ]:
with open('/content/compare.json', 'w') as f:
    json.dump({'before_v2': before, 'after_finetune': after}, f, indent=1)
with zipfile.ZipFile('/content/finetune_result.zip', 'w') as z:
    z.write(best_pt, 'best_finetuned.pt')
    z.write('/content/compare.json', 'compare.json')
    res = os.path.join(os.path.dirname(os.path.dirname(best_pt)), 'results.csv')
    if os.path.exists(res):
        z.write(res, 'results.csv')
files.download('/content/finetune_result.zip')